1. 多个函数调用使用 `for` 循环遍历处理
2. 每个工具调用必须传递自身对应的 `tool_call_id`
3. 借助 `while True` 实现自动多轮调用
    - 执行工具并返回结果
    - 再次请求模型继续处理
    - 直至模型无工具调用需求，输出最终回答
4. 原生支持一次性同时调用 N 个不同函数

### 多函数fucntioncall

In [1]:
from langchain_core.messages import  ToolMessage
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
import json,requests
import muti_utils
from airplane_function_tools import *
load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv("DASHSCOPE_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("BAILIAN_BASE_URL")
print(os.getenv("OPENAI_BASE_URL"))
init_model = init_chat_model(model_provider="openai", model="qwen3.5-flash", api_key=os.getenv("OPENAI_API_KEY"),
                        base_url=os.getenv("OPENAI_BASE_URL"), )
model = init_model.bind_tools(tools)

def main():
    messages = [
        {"role": "system", "content": "你是一个AI小助手。"},
        {"role": "user", "content": "帮我查询2024年4月2日，郑州到北京的航班的票价"}
    ]
     # 🔁 自动多轮调用，直到没有工具需要执行为止
    while True:
        model_response = model.invoke(messages)
        
        if not model_response.tool_calls:
            print("\n✅ 最终回答：")
            print(messages)
            print(model_response.content)
            break
     # 🔥 多个工具调用，逐个执行，逐个返回 tool_call_id
        for tool in model_response.tool_calls:
            tool_id = tool["id"]
            function_name = tool["name"]
            args = tool["args"]
            function_result = ""
            if function_name == "get_plane_number":
                function_result = muti_utils.get_plane_number(**args)
            elif function_name == "get_ticket_price":
                function_result = muti_utils.get_ticket_price(**args)
            else:
                {"error":"未知函数"}
            messages.append(model_response)
            messages.append(ToolMessage(
                tool_call_id = tool_id,
                content= function_result
            ))


if __name__ == '__main__':
    main()

https://dashscope.aliyuncs.com/compatible-mode/v1
查询日期: 2024年4月2日, 航班号: 1123

✅ 最终回答：
[{'role': 'system', 'content': '你是一个AI小助手。'}, {'role': 'user', 'content': '帮我查询2024年4月2日，郑州到北京的航班的票价'}, AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 190, 'prompt_tokens': 431, 'total_tokens': 621, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 128, 'rejected_prediction_tokens': None, 'text_tokens': 190}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'text_tokens': 431}}, 'model_provider': 'openai', 'model_name': 'qwen3.5-flash', 'system_fingerprint': None, 'id': 'chatcmpl-f65f8f85-f0c4-974d-af6d-3735cb93e2ed', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e466d-54b0-73f1-8e01-451d661f4a01-0', tool_calls=[{'name': 'get_plane_number', 'args': {'start_location': '郑州', 'end_location': '北京', 'date': '2024年4月2日'}, 'id': 'call_887be00f3